In [3]:
import pandas as pd
import random
import os

In [4]:
# setup paths
audio_folder = "../raw_data/audio_and_txt_files/"
diagnosis_path = "../raw_data/patient_diagnosis.csv"

In [7]:
# load diagnosis
df_diag = pd.read_csv(diagnosis_path, names=['patient_id', 'diagnosis'])
df_diag['patient_id'] = df_diag['patient_id'].astype(str)

# count how many audio files each patient actually has in the folder
all_files = [f for f in os.listdir(audio_folder) if f.endswith('.wav')]
file_counts = pd.Series([f.split('_')[0] for f in all_files]).value_counts().reset_index()
file_counts.columns = ['patient_id', 'audio_count']

print(file_counts.head())


  patient_id  audio_count
0        130           66
1        107           28
2        151           28
3        138           27
4        172           27


In [13]:
# merge info: diagn + audio count
df_stats = pd.merge(df_diag, file_counts, on='patient_id')

print(df_stats.head())

  patient_id diagnosis  audio_count
0        101      URTI            2
1        102   Healthy            1
2        103    Asthma            1
3        104      COPD            6
4        105      URTI            1


In [12]:
# logic selection
categories = ['COPD', 'Pneumonia', 'Healthy']
demo_blacklist = []

print(f"{'category':<12} | {'ID':<10} | {'audios lost'}")
print("-" * 35)

for cat in categories:
    # filter by cat
    subset = df_stats[df_stats['diagnosis'] == cat]
    
    if not subset.empty:
        # find the min num of audios a patient has in this cat
        min_audios = subset['audio_count'].min()
        
        # get all patients that have this minimun num
        candidates = subset[subset['audio_count'] == min_audios]['patient_id'].tolist()
        
        # pick one randomly
        chosen_id = random.choice(candidates)
        demo_blacklist.append(chosen_id)
        
        print(f"{cat:<12} | {chosen_id:<10} | {min_audios}")

print("\ncopy and paste this into your script preprocessing xgboost!!")
print(f"demo_blacklist = {demo_blacklist}")

category     | ID         | audios lost
-----------------------------------
COPD         | 142        | 1
Pneumonia    | 191        | 3
Healthy      | 182        | 1

copy and paste this into your script preprocessing xgboost!!
demo_blacklist = ['142', '191', '182']
